## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os

# Constants

In [ ]:
FEATURE_COLS = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
MODELS_DIR = "../target/models"
MODEL_PICKLE_FILENAME="model_sklearn.pkl"
SCALER_JOBLIB_FILENAME="scaler_sklearn.joblib"
INPUT_PATH = '../inference/input/input.csv'
INDEXER_PATH = '../target/prepared/label_encoder_species.pkl'
INFERENCE_DIR = "../target/inference"
OUTPUT_CSV = 'output_sklearn.csv'
EXPECTED_CSV = 'expected.csv'
EXPECTED_DIR = '../inference/expected'


## Load the Pickle Model

In [ ]:
models_dir=MODELS_DIR
model_pickle_filename=MODEL_PICKLE_FILENAME
model_path = os.path.join(models_dir, model_pickle_filename)

with open(model_path, 'rb') as f:
    model = pickle.load(f)
print(f'Model loaded from: {model_path}')

## Load Inference Input Data

In [ ]:
input_path = INPUT_PATH
df_input = pd.read_csv(input_path)
print(f'Input data shape: {df_input.shape}')
df_input.head()

In [ ]:
# Convert columns to float
df_input[FEATURE_COLS] = df_input[FEATURE_COLS].astype(float)

In [ ]:
# Display the encoded columns
print("Columns after:")
print(df_input.columns.tolist())

In [ ]:
df_input

# Load Scaler

In [ ]:
import joblib

models_dir=MODELS_DIR
scaler_joblib_filename=SCALER_JOBLIB_FILENAME

scaler_path = os.path.join(models_dir, scaler_joblib_filename)
scaler = joblib.load(scaler_path)

# Load Indexes

In [ ]:
import joblib
indexer_path = INDEXER_PATH
# To convert back during inference:
le = joblib.load(indexer_path)


## Prepare Features for Inference

In [ ]:
# Adjust feature columns as needed to match training
feature_cols = FEATURE_COLS
X_infer = df_input[feature_cols].values.astype(np.float32)
print(f'Inference features shape: {X_infer.shape}')

# standardize the features

In [ ]:
x_infer_scaled = scaler.transform(X_infer)

## Run Inference

In [ ]:
y_pred = model.predict(x_infer_scaled)
print('Predictions:', y_pred)

original_species = le.inverse_transform(y_pred)
print('Original Species:', original_species)

# Verify

In [ ]:
expected_csv = EXPECTED_CSV
expected_dir = EXPECTED_DIR
expected_path = os.path.join(expected_dir, expected_csv)

df_expected = pd.read_csv(expected_path)
expected_species = df_expected['species'].to_numpy(dtype=str)
orig = np.asarray(original_species, dtype=str)

if np.array_equal(orig, expected_species):
    print('Verification: correct — predictions match expected.csv')
else:
    print('Verification: incorrect — predictions do not match expected.csv')
    print('Expected:', expected_species)
    print('Got:', orig)

## Save Predictions to CSV

### Prepare Inference directory

In [ ]:
inference_dir = INFERENCE_DIR
os.makedirs(inference_dir, exist_ok=True)

In [ ]:
output_csv = os.path.join(inference_dir, OUTPUT_CSV)
df_output = df_input.copy()
df_output['Prediction'] = y_pred
df_output['Species'] = original_species
df_output.to_csv(output_csv, index=False)
print(f'Predictions saved to: {output_csv}')